In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Parse timestamps to UTC datetimes as required
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Ensure 'value' columns are numeric where present
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Compute summaries: group by cmdb_id and KPI/trace/log name, count rows and global P95 of 'value'
metric_summary = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(count=('value', 'size'), p95=('value', lambda x: x.quantile(0.95)))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(50)
)
metric_summary['p95'] = metric_summary['p95'].round(6)

trace_summary = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(count=('value', 'size'), p95=('value', lambda x: x.quantile(0.95)))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(50)
)
trace_summary['p95'] = trace_summary['p95'].round(6)

log_summary = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(count=('value', 'size'), p95=('value', lambda x: x.quantile(0.95)))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(50)
)
log_summary['p95'] = log_summary['p95'].round(6)

# error_logs counts per cmdb_id
error_log_counts = (
    error_df
    .groupby('cmdb_id', dropna=False)
    .agg(count=('message', 'size'))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(50)
)

# Display the four compact summaries
metric_summary, trace_summary, log_summary, error_log_counts

```
Out[1]:
```
summary = (
    "Summary of telemetry (compact):\n"
    "- Metrics: adservice appears frequently across many KPIs (count=25 each). Notable metric P95s: adservice.diskio P95 ≈ 7.17e4 (very high), adservice.cpu P95 ≈ 0.776, "
    "paymentservice.mem P95 ≈ 4.52e7 (very large memory value). Many services have 25-sample series; these are the standout high-value metrics.\n"
    "- Traces: checkoutservice shows prominent traces (e.g., duration_mean ≈ 0.0469, duration_p95 ≈ 0.303). There are also higher root/front-end related durations (e.g., trace.from_frontendservice.duration_mean ≈ 0.450). "
    "Trace error_rate entries are mostly 0 in the top rows shown.\n"
    "- Logs: log.error_count is 0 for many services in the top rows (e.g., adservice, cartservice, checkoutservice). log.row_count is large for some services (e.g., cartservice ~793.2, adservice ~465.0).\n"
    "- Error logs: frontend has the largest error_logs count (179), making it the top source of logged error messages.\n\n"
    "Top candidates for further investigation based on these summaries: high disk I/O on adservice, high memory on paymentservice, and the frontend due to a large number of error log entries. "
    "Trace latencies in checkoutservice and front-end related traces are also worth inspecting. These are observations from the provided summaries, not definitive root-cause conclusions."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id    kpi_name  count           p95
0        adservice         cpu     25  7.758130e-01
1        adservice      diskio     25  7.174616e+04
2        adservice       error     25  0.000000e+00
3        adservice  latency-50     25  3.799000e-03
4        adservice  latency-90     25  1.259400e-02
..             ...         ...    ...           ...
45  paymentservice  latency-50     25  3.823000e-03
46  paymentservice  latency-90     25  8.449000e-03
47  paymentservice         mem     25  4.518108e+07
48  paymentservice      socket     25  3.000000e+00
49  paymentservice    workload     25  9.880800e-01

[50 rows x 4 columns],             cmdb_id                                trace_name  count         p95
0   checkoutservice  trace.from_checkoutservice.duration_mean     25    0.046905
1   checkoutservice   trace.from_checkoutservice.duration_p95     25    0.303072
2   checkoutservice     trace.from_checkoutservice.error_rate     25    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count     25  269.600000
4   checkoutservice  trace.from_frontendservice.duration_mean     25    0.450255
..              ...                                       ...    ...         ...
45     emailservice   trace.from_checkoutservice.duration_p95     25    0.000840
46     emailservice     trace.from_checkoutservice.error_rate     25    0.000000
47     emailservice      trace.from_checkoutservice.row_count     25   30.400000
48     emailservice             trace.from_root.duration_mean     25    0.000058
49     emailservice              trace.from_root.duration_p95     25    0.000096

[50 rows x 4 columns],                   cmdb_id         log_name  count    p95
0               adservice  log.error_count     25    0.0
1               adservice    log.row_count     25  465.0
2             cartservice  log.error_count     25    0.0
3             cartservice    log.row_count     25  793.2
4         checkoutservice  log.error_count     25    0.0
..                    ...              ...    ...    ...
21        shippingservice    log.row_count     25  485.2
19                  redis    log.row_count      5    5.0
18                  redis  log.error_count      5    0.0
14  productcatalogservice  log.error_count      4    0.0
15  productcatalogservice    log.row_count      4    3.0

[22 rows x 4 columns],     cmdb_id  count
0  frontend    179)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
# reuse metric_df if present, otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) compute global P95 per (cmdb_id, kpi_name) using all available metric rows
global_p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) define incident window (UTC)
start = pd.to_datetime("2024-01-20 02:03:17", utc=True)
end   = pd.to_datetime("2024-01-20 02:33:17", utc=True)

# 3) filter metric rows within the window
window_df = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 4) attach global_p95 to window rows to detect anomalies (value > global_p95)
if not window_df.empty:
    window_with_p95 = window_df.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')
    window_with_p95['is_anomaly'] = (window_with_p95['value'] > window_with_p95['global_p95']).astype(int)
    # aggregate window stats per series
    agg_window = (
        window_with_p95
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(
            rows_in_window=('value', 'size'),
            count_anomalies_where_value_gt_global_p95=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp_in_window=('timestamp', lambda x: x[window_with_p95.loc[x.index, 'is_anomaly'] == 1].min() if x.size>0 else pd.NaT),
            max_value_in_window=('value', 'max')
        )
        .reset_index()
    )
else:
    # no rows in window -> create empty aggregation
    agg_window = pd.DataFrame(columns=['cmdb_id','kpi_name','rows_in_window','count_anomalies_where_value_gt_global_p95','earliest_anomaly_timestamp_in_window','max_value_in_window'])

# 5) Ensure all series are present by merging global_p95_df with agg_window (fill missing with zeros/NaN)
metric_incident_summary = (
    global_p95_df
    .merge(agg_window, on=['cmdb_id', 'kpi_name'], how='left')
)

# fill missing numeric counts with 0, keep NaN for timestamps and max_value
metric_incident_summary['rows_in_window'] = metric_incident_summary['rows_in_window'].fillna(0).astype(int)
metric_incident_summary['count_anomalies_where_value_gt_global_p95'] = metric_incident_summary['count_anomalies_where_value_gt_global_p95'].fillna(0).astype(int)
# format earliest anomaly timestamp as ISO UTC string or keep NaN
metric_incident_summary['earliest_anomaly_timestamp_in_window'] = pd.to_datetime(metric_incident_summary['earliest_anomaly_timestamp_in_window'])
metric_incident_summary['earliest_anomaly_timestamp_in_window'] = metric_incident_summary['earliest_anomaly_timestamp_in_window'].dt.tz_convert('UTC')
metric_incident_summary['earliest_anomaly_timestamp_in_window'] = metric_incident_summary['earliest_anomaly_timestamp_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
metric_incident_summary['earliest_anomaly_timestamp_in_window'] = metric_incident_summary['earliest_anomaly_timestamp_in_window'].where(~metric_incident_summary['earliest_anomaly_timestamp_in_window'].isna(), pd.NA)

# max_value_in_window stays NaN if no rows
metric_incident_summary['max_value_in_window'] = metric_incident_summary['max_value_in_window'].astype(float)

# round numeric columns for compactness
metric_incident_summary['global_p95'] = metric_incident_summary['global_p95'].round(6)
metric_incident_summary['max_value_in_window'] = metric_incident_summary['max_value_in_window'].round(6)

# 6) sort by anomalies desc and return top 20 rows compactly
metric_incident_summary = metric_incident_summary.sort_values(
    by=['count_anomalies_where_value_gt_global_p95', 'rows_in_window'],
    ascending=[False, False]
).reset_index(drop=True)

metric_incident_summary.head(20)

```
Out[2]:
```
summary = (
    "Incident window analysis (2024-01-20 02:03:17 to 02:33:17 UTC):\n"
    "- Multiple series exceeded their global P95 during the window. The top affected services are adservice, cartservice, checkoutservice, and currencyservice.\n"
    "- Each listed series shows 25 rows in-window and 2 values above its global P95 (count_anomalies = 2).\n"
    "- adservice is the most notable: CPU, diskio, memory, latency-50/90 and workload all had anomalies. "
    "Earliest adservice anomaly appears at 2024-01-20T02:06:00Z (diskio) and CPU anomaly at 2024-01-20T02:11:00Z. Maxs include diskio ~7.48e4 and mem ~1.14e8.\n"
    "- cartservice, checkoutservice, and currencyservice show similar patterns (CPU, mem, latency, workload) with earliest anomalies between ~02:06 and ~02:27 UTC.\n"
    "Conclusion: adservice is the top candidate for root-cause investigation (multiple KPI spikes), with cartservice/checkoutservice/currencyservice as secondary suspects. "
    "Recommend investigating adservice resource usage (disk I/O and memory) and related latency during the noted timestamps."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    global_p95  rows_in_window  count_anomalies_where_value_gt_global_p95 earliest_anomaly_timestamp_in_window  max_value_in_window
0         adservice         cpu  7.758130e-01              25                                          2                 2024-01-20T02:11:00Z         7.825810e-01
1         adservice      diskio  7.174616e+04              25                                          2                 2024-01-20T02:06:00Z         7.476956e+04
2         adservice  latency-50  3.799000e-03              25                                          2                 2024-01-20T02:20:00Z         3.951000e-03
3         adservice  latency-90  1.259400e-02              25                                          2                 2024-01-20T02:20:00Z         1.364300e-02
4         adservice         mem  1.143820e+08              25                                          2                 2024-01-20T02:20:00Z         1.144784e+08
5         adservice    workload  1.531640e+01              25                                          2                 2024-01-20T02:06:00Z         1.564051e+01
6       cartservice         cpu  2.897011e+00              25                                          2                 2024-01-20T02:06:00Z         2.998462e+00
7       cartservice  latency-50  3.834000e-03              25                                          2                 2024-01-20T02:23:00Z         4.074000e-03
8       cartservice  latency-90  9.171000e-03              25                                          2                 2024-01-20T02:21:00Z         6.062200e-02
9       cartservice         mem  4.371614e+07              25                                          2                 2024-01-20T02:23:00Z         4.403261e+07
10      cartservice    workload  2.615958e+01              25                                          2                 2024-01-20T02:14:00Z         2.626888e+01
11  checkoutservice         cpu  4.574160e-01              25                                          2                 2024-01-20T02:15:00Z         4.896980e-01
12  checkoutservice  latency-50  3.739170e-01              25                                          2                 2024-01-20T02:27:00Z         3.886880e-01
13  checkoutservice  latency-90  1.357486e+00              25                                          2                 2024-01-20T02:22:00Z         1.768071e+00
14  checkoutservice         mem  1.240887e+07              25                                          2                 2024-01-20T02:16:00Z         1.243150e+07
15  checkoutservice    workload  1.010850e+00              25                                          2                 2024-01-20T02:15:00Z         1.092183e+00
16  currencyservice         cpu  3.406910e+01              25                                          2                 2024-01-20T02:06:00Z         3.461330e+01
17  currencyservice  latency-50  1.287300e-02              25                                          2                 2024-01-20T02:21:00Z         1.369800e-02
18  currencyservice  latency-90  9.836900e-02              25                                          2                 2024-01-20T02:21:00Z         1.051960e-01
19  currencyservice         mem  4.571121e+07              25                                          2                 2024-01-20T02:16:00Z         4.585854e+07```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse trace_df if present, otherwise load and prepare
if 'trace_df' not in globals():
    trace_df = pd.read_csv('trace.csv')
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
else:
    # ensure timestamp and value are correctly typed if already in memory
    if trace_df.get('timestamp').dtype == 'int64' or trace_df.get('timestamp').dtype == 'float64':
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) compute global P95 per (cmdb_id, trace_name) using all available trace rows (global threshold)
global_p95_df = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) define incident window (UTC)
start = pd.to_datetime("2024-01-20 02:03:17", utc=True)
end   = pd.to_datetime("2024-01-20 02:33:17", utc=True)

# 3) filter trace rows within the window
window_df = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 4) attach global_p95 to window rows and detect anomalies (value > global_p95)
if not window_df.empty:
    window_with_p95 = window_df.merge(global_p95_df, on=['cmdb_id', 'trace_name'], how='left')
    window_with_p95['is_anomaly'] = (window_with_p95['value'] > window_with_p95['global_p95']).astype(int)

    # aggregate basic window stats
    agg_basic = (
        window_with_p95
        .groupby(['cmdb_id', 'trace_name'], dropna=False)
        .agg(
            rows_in_window=('value', 'size'),
            count_anomalies_where_value_gt_global_p95=('is_anomaly', 'sum'),
            max_value_in_window=('value', 'max')
        )
        .reset_index()
    )

    # earliest anomaly timestamp per group (only where is_anomaly==1)
    earliest_anom = (
        window_with_p95[window_with_p95['is_anomaly'] == 1]
        .groupby(['cmdb_id', 'trace_name'], dropna=False)['timestamp']
        .min()
        .reset_index(name='earliest_anomaly_timestamp_in_window')
    )

    # merge earliest anomaly timestamp into agg_basic
    agg_window = agg_basic.merge(earliest_anom, on=['cmdb_id', 'trace_name'], how='left')
else:
    # no rows in window -> empty aggregation (but will merge later with global_p95_df)
    agg_window = pd.DataFrame(columns=[
        'cmdb_id', 'trace_name', 'rows_in_window',
        'count_anomalies_where_value_gt_global_p95',
        'max_value_in_window', 'earliest_anomaly_timestamp_in_window'
    ])

# 5) Ensure all trace series are present by merging global_p95_df with agg_window
trace_incident_summary = global_p95_df.merge(agg_window, on=['cmdb_id', 'trace_name'], how='left')

# fill missing numeric counts with 0, keep NaN for timestamps and max_value
trace_incident_summary['rows_in_window'] = trace_incident_summary['rows_in_window'].fillna(0).astype(int)
trace_incident_summary['count_anomalies_where_value_gt_global_p95'] = trace_incident_summary['count_anomalies_where_value_gt_global_p95'].fillna(0).astype(int)
trace_incident_summary['max_value_in_window'] = trace_incident_summary['max_value_in_window'].astype(float)

# format earliest anomaly timestamp as ISO UTC string or keep NA
trace_incident_summary['earliest_anomaly_timestamp_in_window'] = pd.to_datetime(trace_incident_summary['earliest_anomaly_timestamp_in_window'])
trace_incident_summary['earliest_anomaly_timestamp_in_window'] = trace_incident_summary['earliest_anomaly_timestamp_in_window'].dt.tz_convert('UTC')
trace_incident_summary['earliest_anomaly_timestamp_in_window'] = trace_incident_summary['earliest_anomaly_timestamp_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
trace_incident_summary['earliest_anomaly_timestamp_in_window'] = trace_incident_summary['earliest_anomaly_timestamp_in_window'].where(
    ~trace_incident_summary['earliest_anomaly_timestamp_in_window'].isna(), pd.NA
)

# round numeric columns for compactness
trace_incident_summary['global_p95'] = trace_incident_summary['global_p95'].round(6)
trace_incident_summary['max_value_in_window'] = trace_incident_summary['max_value_in_window'].round(6)

# 6) sort by anomaly count desc then rows_in_window desc and return top 20 rows
trace_incident_summary = trace_incident_summary.sort_values(
    by=['count_anomalies_where_value_gt_global_p95', 'rows_in_window'],
    ascending=[False, False]
).reset_index(drop=True)

trace_incident_summary.head(20)

```
Out[3]:
```
summary = (
    "Incident trace analysis (2024-01-20 02:03:17 to 02:33:17 UTC):\n"
    "- Checkoutservice is the clear hotspot: many trace series for checkoutservice (duration_mean, duration_p95, row_count, both from/to frontend and to other services) had rows in-window (25 each) and showed 2 values above their global P95.\n"
    "- Notable observations: trace.from_frontendservice.duration_mean peaked ~0.712s (earliest anomaly 2024-01-20T02:22:00Z), trace.from_frontendservice.duration_p95 peaked ~1.81s (earliest anomaly 02:21:00Z), and row_count series reached up to ~297 (earliest anomaly 02:09:00Z).\n"
    "- Other checkoutservice outbound traces to currency/email/payment also show anomaly occurrences (earliest anomaly timestamps between ~02:07 and ~02:17 UTC).\n"
    "- All affected checkoutservice trace series show exactly 2 anomaly points during the window, indicating intermittent spikes rather than sustained continuous degradation.\n"
    "Conclusion / action: prioritize investigating checkoutservice (latency spikes and traffic/row_count increases) and its interactions with frontend, currency, email, and payment services around the noted timestamps. These traces point to checkoutservice as the primary suspect for the incident."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                trace_name  global_p95  rows_in_window  count_anomalies_where_value_gt_global_p95  max_value_in_window earliest_anomaly_timestamp_in_window
0   checkoutservice  trace.from_checkoutservice.duration_mean    0.046905              25                                          2             0.072050                 2024-01-20T02:22:00Z
1   checkoutservice   trace.from_checkoutservice.duration_p95    0.303072              25                                          2             0.343937                 2024-01-20T02:29:00Z
2   checkoutservice      trace.from_checkoutservice.row_count  269.600000              25                                          2           297.000000                 2024-01-20T02:09:00Z
3   checkoutservice  trace.from_frontendservice.duration_mean    0.450255              25                                          2             0.712391                 2024-01-20T02:22:00Z
4   checkoutservice   trace.from_frontendservice.duration_p95    1.416035              25                                          2             1.809632                 2024-01-20T02:21:00Z
5   checkoutservice      trace.from_frontendservice.row_count   30.400000              25                                          2            36.000000                 2024-01-20T02:09:00Z
6   checkoutservice             trace.from_root.duration_mean    0.018049              25                                          2             0.019008                 2024-01-20T02:14:00Z
7   checkoutservice              trace.from_root.duration_p95    0.076362              25                                          2             0.077829                 2024-01-20T02:23:00Z
8   checkoutservice                 trace.from_root.row_count  140.000000              25                                          2           143.000000                 2024-01-20T02:09:00Z
9   checkoutservice    trace.to_checkoutservice.duration_mean    0.046905              25                                          2             0.072050                 2024-01-20T02:22:00Z
10  checkoutservice     trace.to_checkoutservice.duration_p95    0.303072              25                                          2             0.343937                 2024-01-20T02:29:00Z
11  checkoutservice        trace.to_checkoutservice.row_count  269.600000              25                                          2           297.000000                 2024-01-20T02:09:00Z
12  checkoutservice    trace.to_currencyservice.duration_mean    0.001544              25                                          2             0.002223                 2024-01-20T02:10:00Z
13  checkoutservice     trace.to_currencyservice.duration_p95    0.000424              25                                          2             0.000563                 2024-01-20T02:07:00Z
14  checkoutservice        trace.to_currencyservice.row_count  116.000000              25                                          2           119.000000                 2024-01-20T02:09:00Z
15  checkoutservice       trace.to_emailservice.duration_mean    0.000382              25                                          2             0.000391                 2024-01-20T02:17:00Z
16  checkoutservice        trace.to_emailservice.duration_p95    0.000840              25                                          2             0.000898                 2024-01-20T02:17:00Z
17  checkoutservice           trace.to_emailservice.row_count   30.400000              25                                          2            36.000000                 2024-01-20T02:09:00Z
18  checkoutservice     trace.to_paymentservice.duration_mean    0.000299              25                                          2             0.000459                 2024-01-20T02:13:00Z
19  checkoutservice      trace.to_paymentservice.duration_p95    0.000878              25                                          2             0.002326                 2024-01-20T02:13:00Z```
```

In [ ]:
# In[4]:

import pandas as pd

# reuse existing dataframes if present, otherwise load
if 'log_df' not in globals():
    log_df = pd.read_csv('log.csv')
if 'error_df' not in globals():
    error_df = pd.read_csv('error_logs.csv')

# parse timestamps and ensure numeric values
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime("2024-01-20 02:03:17", utc=True)
end   = pd.to_datetime("2024-01-20 02:33:17", utc=True)

# 1) compute global P95 per (cmdb_id, log_name) using the entire log series (global threshold)
global_p95_log = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) filter log rows inside the incident window
log_window = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()

# 3) attach global_p95 and detect anomalies (value > global_p95) for window rows
if not log_window.empty:
    log_window_with_p95 = log_window.merge(global_p95_log, on=['cmdb_id', 'log_name'], how='left')
    log_window_with_p95['is_anomaly'] = (log_window_with_p95['value'] > log_window_with_p95['global_p95']).astype(int)

    # aggregate per series within window
    agg_basic_log = (
        log_window_with_p95
        .groupby(['cmdb_id', 'log_name'], dropna=False)
        .agg(
            rows_in_window=('value', 'size'),
            count_anomalies_where_value_gt_global_p95=('is_anomaly', 'sum'),
            max_value_in_window=('value', 'max')
        )
        .reset_index()
    )

    # earliest anomaly timestamp per group (only where is_anomaly==1)
    earliest_anom_log = (
        log_window_with_p95[log_window_with_p95['is_anomaly'] == 1]
        .groupby(['cmdb_id', 'log_name'], dropna=False)['timestamp']
        .min()
        .reset_index(name='earliest_anomaly_timestamp_in_window')
    )

    agg_window_log = agg_basic_log.merge(earliest_anom_log, on=['cmdb_id', 'log_name'], how='left')
else:
    agg_window_log = pd.DataFrame(columns=[
        'cmdb_id','log_name','rows_in_window',
        'count_anomalies_where_value_gt_global_p95',
        'max_value_in_window','earliest_anomaly_timestamp_in_window'
    ])

# 4) ensure all series present by merging global_p95 with window aggregation
log_incident_summary = global_p95_log.merge(agg_window_log, on=['cmdb_id', 'log_name'], how='left')

# fill defaults: counts 0 for missing, keep NaN for timestamps and max_value
log_incident_summary['rows_in_window'] = log_incident_summary['rows_in_window'].fillna(0).astype(int)
log_incident_summary['count_anomalies_where_value_gt_global_p95'] = log_incident_summary['count_anomalies_where_value_gt_global_p95'].fillna(0).astype(int)
log_incident_summary['max_value_in_window'] = log_incident_summary['max_value_in_window'].astype(float)

# format earliest anomaly timestamp as ISO UTC string or keep NA
log_incident_summary['earliest_anomaly_timestamp_in_window'] = pd.to_datetime(log_incident_summary['earliest_anomaly_timestamp_in_window'])
log_incident_summary['earliest_anomaly_timestamp_in_window'] = log_incident_summary['earliest_anomaly_timestamp_in_window'].dt.tz_convert('UTC')
log_incident_summary['earliest_anomaly_timestamp_in_window'] = log_incident_summary['earliest_anomaly_timestamp_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
log_incident_summary['earliest_anomaly_timestamp_in_window'] = log_incident_summary['earliest_anomaly_timestamp_in_window'].where(
    ~log_incident_summary['earliest_anomaly_timestamp_in_window'].isna(), pd.NA
)

# round numeric columns for compactness
log_incident_summary['global_p95'] = log_incident_summary['global_p95'].round(6)
log_incident_summary['max_value_in_window'] = log_incident_summary['max_value_in_window'].round(6)

# sort by anomaly count desc then rows_in_window desc and keep top 20
log_incident_summary = log_incident_summary.sort_values(
    by=['count_anomalies_where_value_gt_global_p95', 'rows_in_window'],
    ascending=[False, False]
).reset_index(drop=True)

log_summary = log_incident_summary.head(20)

# -----------------------
# Error logs summary:
# total counts per cmdb_id across the entire error_logs.csv
error_counts_total = (
    error_df
    .groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='total_error_count')
    .sort_values('total_error_count', ascending=False)
)

# earliest error timestamp inside the incident window per cmdb_id (if any)
error_in_window = error_df[(error_df['timestamp'] >= start) & (error_df['timestamp'] <= end)].copy()
if not error_in_window.empty:
    earliest_in_window = (
        error_in_window
        .groupby('cmdb_id', dropna=False)['timestamp']
        .min()
        .reset_index(name='earliest_error_timestamp_in_window')
    )
    # format timestamp to ISO UTC
    earliest_in_window['earliest_error_timestamp_in_window'] = earliest_in_window['earliest_error_timestamp_in_window'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
else:
    earliest_in_window = pd.DataFrame(columns=['cmdb_id', 'earliest_error_timestamp_in_window'])

# merge total counts with earliest-in-window info
error_log_counts_summary = error_counts_total.merge(earliest_in_window, on='cmdb_id', how='left')

# where no earliest timestamp, keep NA
error_log_counts_summary['earliest_error_timestamp_in_window'] = error_log_counts_summary['earliest_error_timestamp_in_window'].where(
    ~error_log_counts_summary['earliest_error_timestamp_in_window'].isna(), pd.NA
)

# keep top 20 rows by total_error_count
error_log_counts_summary = error_log_counts_summary.head(20)

# Final compact outputs (DataFrames)
log_summary, error_log_counts_summary

```
Out[4]:
```
summary = (
    "Log and error-summary (incident window 2024-01-20 02:03:17 to 02:33:17 UTC):\n"
    "- Many services experienced spikes in log row_count during the window (each with 25 rows in-window and 2 points above their global P95):\n"
    "  cartservice (row_count peak 806.0, earliest anomaly 2024-01-20T02:12:00Z),\n"
    "  checkoutservice (row_count peak 108.0, earliest 2024-01-20T02:09:00Z),\n"
    "  currencyservice (row_count peak 2550.0, earliest 2024-01-20T02:11:00Z),\n"
    "  emailservice (row_count peak 36.0, earliest 2024-01-20T02:09:00Z),\n"
    "  frontend (row_count peak 2826.0, earliest 2024-01-20T02:11:00Z),\n"
    "  paymentservice, recommendationservice, shippingservice also show row_count spikes (earliest anomalies around 02:09–02:12 UTC).\n"
    "- frontend.log.error_count also shows anomalies (2 points above global P95; max 107.0) with earliest error-count anomaly at 2024-01-20T02:25:00Z.\n"
    "- adservice had a single row_count anomaly (earliest 2024-01-20T02:17:00Z) and no log.error_count anomalies in the window.\n"
    "- Error log file (full): frontend has by far the most error lines (total 179). The earliest error inside the incident window for frontend is 2024-01-20T02:25:26Z.\n\n"
    "Interpretation / recommended next steps:\n"
    "- The widespread row_count spikes point to increased traffic or processing load affecting cart, checkout, currency, frontend and related services around 02:09–02:12 UTC.\n"
    "- The frontend shows both high row_count and many error log lines (179), with error occurrences inside the window (earliest 02:25:26Z). Investigate frontend errors and the traffic surge pattern first.\n"
    "- Also inspect cartservice/checkoutservice/currencyservice for processing/backpressure issues (their row_count spikes and earliest anomaly timestamps listed above).\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id         log_name  global_p95  rows_in_window  count_anomalies_where_value_gt_global_p95  max_value_in_window earliest_anomaly_timestamp_in_window
0             cartservice    log.row_count       793.2              25                                          2                806.0                 2024-01-20T02:12:00Z
1         checkoutservice    log.row_count        91.2              25                                          2                108.0                 2024-01-20T02:09:00Z
2         currencyservice    log.row_count      2491.4              25                                          2               2550.0                 2024-01-20T02:11:00Z
3            emailservice    log.row_count        30.4              25                                          2                 36.0                 2024-01-20T02:09:00Z
4                frontend  log.error_count        57.6              25                                          2                107.0                 2024-01-20T02:25:00Z
5                frontend    log.row_count      2744.4              25                                          2               2826.0                 2024-01-20T02:11:00Z
6          paymentservice    log.row_count        60.8              25                                          2                 72.0                 2024-01-20T02:09:00Z
7   recommendationservice    log.row_count       577.0              25                                          2                582.0                 2024-01-20T02:12:00Z
8         shippingservice    log.row_count       485.2              25                                          2                490.0                 2024-01-20T02:09:00Z
9               adservice    log.row_count       465.0              25                                          1                466.0                 2024-01-20T02:17:00Z
10              adservice  log.error_count         0.0              25                                          0                  0.0                                 <NA>
11            cartservice  log.error_count         0.0              25                                          0                  0.0                                 <NA>
12        checkoutservice  log.error_count         0.0              25                                          0                  0.0                                 <NA>
13        currencyservice  log.error_count         0.0              25                                          0                  0.0                                 <NA>
14           emailservice  log.error_count         0.0              25                                          0                  0.0                                 <NA>
15         paymentservice  log.error_count         0.0              25                                          0                  0.0                                 <NA>
16  recommendationservice  log.error_count         0.0              25                                          0                  0.0                                 <NA>
17        shippingservice  log.error_count         0.0              25                                          0                  0.0                                 <NA>
18                  redis  log.error_count         0.0               5                                          0                  0.0                                 <NA>
19                  redis    log.row_count         5.0               5                                          0                  5.0                                 <NA>,     cmdb_id  total_error_count earliest_error_timestamp_in_window
0  frontend                179               2024-01-20T02:25:26Z)```
```